# Common NCET Failures and Exceptions

This notebook intentionally triggers several common NCET errors and then shows the corresponding correction. Expected exceptions are caught so that the notebook can be run from top to bottom.

NCET rejects these cases instead of silently approximating the PyTorch model. This preserves the exactness contract.

In [ ]:
from __future__ import annotations

from collections.abc import Callable

import numpy as np
import torch
from torch import nn

from ncet import (
    Bounds,
    GraphCaptureError,
    InvalidBoundsError,
    UnsupportedOperatorError,
    form_milp,
)


def expect_error(
    expected: type[Exception],
    action: Callable[[], object],
) -> None:
    """Run one failing example and verify its public exception type."""
    try:
        action()
    except expected as error:
        print(f"{type(error).__name__}: {error}")
    except Exception as error:
        raise AssertionError(
            f"expected {expected.__name__}, got {type(error).__name__}"
        ) from error
    else:
        raise AssertionError(f"expected {expected.__name__}, but no error was raised")

## 1. Invalid input bounds

Every element must satisfy lower <= upper. Reversed bounds create an empty input domain, so NCET raises InvalidBoundsError before constructing the optimization model.

In [ ]:
class TinyLinear(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


linear_model = TinyLinear().eval()
invalid_bounds = Bounds(
    lower=np.array([0.0, 0.0]),
    upper=np.array([-1.0, 1.0]),
)
expect_error(
    InvalidBoundsError,
    lambda: form_milp(linear_model, invalid_bounds),
)

# Correction: provide a nonempty elementwise interval.
linear_encoding = form_milp(
    linear_model,
    Bounds(lower=np.full(2, -1.0), upper=np.full(2, 1.0)),
)
print("corrected encoding:", linear_encoding.stats)

## 2. Unsupported nonlinear operator

Sigmoid is not in NCET's exact operator boundary, so it raises UnsupportedOperatorError. Use a supported operator only when it is appropriate for the intended model; replacing Sigmoid with ReLU changes the network semantics.

In [ ]:
class SigmoidNetwork(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(x)


vector_bounds = Bounds(lower=np.full(2, -1.0), upper=np.full(2, 1.0))
expect_error(
    UnsupportedOperatorError,
    lambda: form_milp(SigmoidNetwork().eval(), vector_bounds),
)


class SupportedActivationNetwork(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(x)


relu_encoding = form_milp(SupportedActivationNetwork().eval(), vector_bounds)
print("supported alternative:", relu_encoding.stats)

## 3. In-place tensor mutation

FX graphs used by NCET must be functional: an operation produces a new symbolic value instead of overwriting an existing one. Therefore nn.ReLU(inplace=True) raises GraphCaptureError.

In [ ]:
class InplaceReLU(nn.Module):
    def __init__(self, inplace: bool) -> None:
        super().__init__()
        self.relu = nn.ReLU(inplace=inplace)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(x)


expect_error(
    GraphCaptureError,
    lambda: form_milp(InplaceReLU(inplace=True).eval(), vector_bounds),
)

# Correction: use the out-of-place form.
out_of_place = form_milp(InplaceReLU(inplace=False).eval(), vector_bounds)
print("corrected encoding:", out_of_place.stats)

## 4. Tensor-dependent Python control flow

fx.symbolic_trace() cannot decide a Python if condition whose value depends on a symbolic tensor. NCET reports this as GraphCaptureError. Redesign the model as a static graph composed of supported tensor operations.

In [ ]:
class DynamicBranch(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.sum() > 0:
            return x
        return -x


expect_error(
    GraphCaptureError,
    lambda: form_milp(DynamicBranch().eval(), vector_bounds),
)


class StaticResidual(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(x) + x


static_encoding = form_milp(StaticResidual().eval(), vector_bounds)
print("static-graph alternative:", static_encoding.stats)

## 5. Unsupported layer configuration

Supporting an operator name does not imply support for every PyTorch option. NCET currently requires Conv2d.groups == 1; grouped or depthwise convolution raises UnsupportedOperatorError.

In [ ]:
class SmallConv(nn.Module):
    def __init__(self, groups: int) -> None:
        super().__init__()
        self.conv = nn.Conv2d(2, 2, kernel_size=1, groups=groups)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


image_bounds = Bounds(
    lower=np.full((2, 3, 3), -1.0),
    upper=np.full((2, 3, 3), 1.0),
)
expect_error(
    UnsupportedOperatorError,
    lambda: form_milp(SmallConv(groups=2).eval(), image_bounds),
)

# Correction: use a standard convolution within the current support boundary.
conv_encoding = form_milp(SmallConv(groups=1).eval(), image_bounds)
print("corrected encoding:", conv_encoding.stats)

## Troubleshooting summary

| Exception | First thing to inspect |
| --- | --- |
| InvalidBoundsError | Bounds shapes, input names/order, finite values, and lower <= upper |
| GraphCaptureError | In-place mutation and tensor-dependent Python control flow |
| UnsupportedOperatorError | The canonical operator and its supported options |
| ExactnessContractError | Static shapes, indices, axes, and other exactness assumptions |

Treat these exceptions as model/interface diagnostics. Do not catch an error and continue with a silently modified encoding.